In [1]:
!pip install -qU langchain langchain-groq wikipedia langchain-community

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.2/504.2 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter

In [2]:
from kaggle_secrets import UserSecretsClient
from langchain_groq import ChatGroq
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents import create_agent
import os
import csv
import time
import string

adversarial_dataset = [
    {
        "question": "Who is the wife of the actor who played 'Neo' in The Matrix, and what is her birth year?",
        "answer": "alexandra grant"
    },
    {
        "question": "Who was the childhood best friend of the oldest person alive in 1850?",
        "answer": "unavailable" 
    },
    {
        "question": "What is the name of the fourth studio album released by the band that wrote 'Stairway to Heaven', and what month was it released?",
        "answer": "november"
    },
    {
        "question": "Who was the mayor of the lost city of Atlantis during the year 1492?",
        "answer": "unavailable"
    },
    {
        "question": "What profession does Nicholas Ray and Elia Kazan have in common?",
        "answer": "director"
    },
    {
        "question": "Which magazine was started first, Arthur's Magazine or First for Women?",
        "answer": "arthur's magazine"
    }
]

secret_label = "GROQ_API_KEY"
secret_value = UserSecretsClient().get_secret(secret_label)

llm = ChatGroq(
    model_name="llama-3.1-8b-instant", 
    temperature=0,
    api_key=secret_value,
    model_kwargs={"parallel_tool_calls": False}
)

api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=500
)

wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)
tools = [wiki_tool]

def normalize_answer(text):
    return text.lower().translate(str.maketrans('', '', string.punctuation)).strip()

In [3]:
# This is evaluate_cot.py
# Without external tools, the LLM will confidently hallucinate answers to multi-hop or impossible questions

results = []

print("\n🚀 Starting CoT (No Tools) Evaluation Pipeline...\n")

for i, data in enumerate(adversarial_dataset):
    question = data["question"]
    ground_truth = data["answer"]

    print(f"\n--- Question {i+1}/{len(adversarial_dataset)} ---")
    print(f"Q: {question}")

    predicted_answer = "AGENT FAILED"
    error_status = "None"

    try:
        response = llm.invoke(
            f"Think step-by-step to answer this question. "
            f"If the answer does not exist, reply 'unavailable'. Question: {question}"
        )
        predicted_answer = response.content
        print(f"💭 Thought/Answer: {predicted_answer}")

    except Exception as e:
        error_status = str(e)
        print(f"❌ Error: {error_status}")

    norm_truth = normalize_answer(ground_truth)
    norm_pred = normalize_answer(predicted_answer)

    truth_words = set(norm_truth.split())
    pred_words = set(norm_pred.split())
    exact_match = 1 if truth_words.issubset(pred_words) and predicted_answer != "AGENT FAILED" else 0

    print(f"\nGround Truth: {ground_truth}")
    print(f"EM Score: {exact_match}")

    results.append({
        "Question": question,
        "Ground Truth": ground_truth,
        "Predicted": predicted_answer,
        "Exact Match": exact_match,
        "Error": error_status
    })
    
    time.sleep(2)

csv_filename = "cot_results.csv"

with open(csv_filename, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["Question", "Ground Truth", "Predicted", "Exact Match", "Error"]
    )
    writer.writeheader()
    writer.writerows(results)

total = len(results)
correct = sum(r["Exact Match"] for r in results)
accuracy = correct / total

print("\n✅ Evaluation Complete!")
print(f"📊 CoT Accuracy: {accuracy:.2f}")
print(f"📁 Results saved to {csv_filename}")


🚀 Starting CoT (No Tools) Evaluation Pipeline...


--- Question 1/6 ---
Q: Who is the wife of the actor who played 'Neo' in The Matrix, and what is her birth year?
💭 Thought/Answer: To answer this question, I will follow these steps:

1. Identify the actor who played 'Neo' in The Matrix.
2. Find information about the actor's wife.
3. Determine the birth year of the actor's wife.

Step 1: Identify the actor who played 'Neo' in The Matrix.
The actor who played 'Neo' in The Matrix is Keanu Reeves.

Step 2: Find information about the actor's wife.
Keanu Reeves has been married to Alexandra Grant since 2022. However, before that, he was in a long-term relationship with Jennifer Syme, but they were not married. 

Step 3: Determine the birth year of the actor's wife.
Since Keanu Reeves is married to Alexandra Grant, I will look for her birth year. Alexandra Grant was born in 1973.

So, the answer to the question is: Alexandra Grant, born in 1973.

Ground Truth: alexandra grant
EM Score: 1

-

In [5]:
# evaluate_baseline.py
# It introduces the Wikipedia tool but lacks memory. This will demonstrate the infinite search loops when it encounters the "unavailable" trap questions.

agent_executor = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful reasoning agent. You must always use the "
        "wikipedia tool to search for factual information before answering."
    )
)

results = []

print("\n🚀 Starting Baseline ReAct Evaluation Pipeline...\n")

for i, data in enumerate(adversarial_dataset):
    question = data["question"]
    ground_truth = data["answer"]

    print(f"\n--- Question {i+1}/{len(adversarial_dataset)} ---")
    print(f"Q: {question}")

    predicted_answer = "AGENT FAILED"
    error_status = "None"

    try:
        response = agent_executor.invoke({"messages": [("user", question)]})
        
        for msg in response["messages"][1:]:
            if msg.type == "ai":
                if msg.content:
                    print(f"💭 Thought/Answer: {msg.content}")
                    predicted_answer = msg.content
                if getattr(msg, "tool_calls", None):
                    for tc in msg.tool_calls:
                        print(f"🛠️  Action: {tc['name']} -> {tc['args']}")
            elif msg.type == "tool":
                print(f"👁️  Observation: {msg.content[:200]}...")

    except Exception as e:
        error_status = str(e)
        print(f"❌ Error: {error_status}")

    norm_truth = normalize_answer(ground_truth)
    norm_pred = normalize_answer(predicted_answer)

    truth_words = set(norm_truth.split())
    pred_words = set(norm_pred.split())
    exact_match = 1 if truth_words.issubset(pred_words) and predicted_answer != "AGENT FAILED" else 0

    print(f"\nGround Truth: {ground_truth}")
    print(f"EM Score: {exact_match}")

    results.append({
        "Question": question,
        "Ground Truth": ground_truth,
        "Predicted": predicted_answer,
        "Exact Match": exact_match,
        "Error": error_status
    })

    time.sleep(2) 

csv_filename = "baseline_results.csv"

with open(csv_filename, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["Question", "Ground Truth", "Predicted", "Exact Match", "Error"]
    )
    writer.writeheader()
    writer.writerows(results)

total = len(results)
correct = sum(r["Exact Match"] for r in results)
accuracy = correct / total

print("\n✅ Evaluation Complete!")
print(f"📊 Baseline Accuracy: {accuracy:.2f}")
print(f"📁 Results saved to {csv_filename}")


🚀 Starting Baseline ReAct Evaluation Pipeline...


--- Question 1/6 ---
Q: Who is the wife of the actor who played 'Neo' in The Matrix, and what is her birth year?
🛠️  Action: wikipedia -> {'query': 'Keanu Reeves wife birth year'}
👁️  Observation: Page: Keanu Reeves
Summary: Keanu Charles Reeves ( kee-AH-noo; born September 2, 1964) is a Canadian actor and musician. The recipient of numerous accolades in a career on screen spanning four decades...
🛠️  Action: wikipedia -> {'query': 'Keanu Reeves partner'}
👁️  Observation: Page: Good Fortune (film)
Summary: Good Fortune is a 2025 American comedy film written and directed by Aziz Ansari in his feature directorial debut. Seth Rogen, Ansari, Keke Palmer, Sandra Oh and Kean...
💭 Thought/Answer: The information provided does not mention Keanu Reeves' wife or partner. However, it is known that Keanu Reeves is in a relationship with artist Alexandra Grant.

Ground Truth: alexandra grant
EM Score: 1

--- Question 2/6 ---
Q: Who was the childho

In [6]:
# evalute_memory.py
# The addition of the memory buffer and the strict system prompt will force the agent to recognize failed searches, pivot, and ultimately conclude if information is genuinely unavailable.

system_prompt = (
    "You are a highly advanced reasoning agent equipped with a conversation memory buffer. "
    "You must always use the wikipedia tool to search for factual information before answering. "
    "CRITICAL RULE: Check your past actions. If a previous tool call failed, returned an error, "
    "or returned irrelevant information, DO NOT repeat the exact same search query. "
    "You must pivot and try a different keyword. "
    "If you cannot find the information after pivoting, state that the information is unavailable."
)

agent_executor = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

results = []

print("\n🚀 Starting MEMORY-AUGMENTED Evaluation Pipeline...\n")

for i, data in enumerate(adversarial_dataset):
    question = data["question"]
    ground_truth = data["answer"]

    print(f"\n--- Question {i+1}/{len(adversarial_dataset)} ---")
    print(f"Q: {question}")

    predicted_answer = "AGENT FAILED"
    error_status = "None"
    
    conversation_buffer_memory = []

    try:
        conversation_buffer_memory.append(("user", question))
        
        response = agent_executor.invoke({"messages": conversation_buffer_memory})
        
        for msg in response["messages"][1:]:
            if msg.type == "ai":
                if msg.content:
                    print(f"💭 Thought/Answer: {msg.content}")
                    predicted_answer = msg.content
                if getattr(msg, "tool_calls", None):
                    for tc in msg.tool_calls:
                        print(f"🛠️  Action: {tc['name']} -> {tc['args']}")
            elif msg.type == "tool":
                print(f"👁️  Observation: {msg.content[:200]}...")

    except Exception as e:
        error_status = str(e)
        print(f"❌ Error: {error_status}")

    norm_truth = normalize_answer(ground_truth)
    norm_pred = normalize_answer(predicted_answer)

    truth_words = set(norm_truth.split())
    pred_words = set(norm_pred.split())
    exact_match = 1 if truth_words.issubset(pred_words) and predicted_answer != "AGENT FAILED" else 0

    print(f"\nGround Truth: {ground_truth}")
    print(f"EM Score: {exact_match}")

    results.append({
        "Question": question,
        "Ground Truth": ground_truth,
        "Predicted": predicted_answer,
        "Exact Match": exact_match,
        "Error": error_status
    })

    time.sleep(2)

csv_filename = "memory_results.csv"

with open(csv_filename, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["Question", "Ground Truth", "Predicted", "Exact Match", "Error"]
    )
    writer.writeheader()
    writer.writerows(results)

total = len(results)
correct = sum(r["Exact Match"] for r in results)
accuracy = correct / total

print("\n✅ Evaluation Complete!")
print(f"📊 Memory-Augmented Accuracy: {accuracy:.2f}")
print(f"📁 Results saved to {csv_filename}")


🚀 Starting MEMORY-AUGMENTED Evaluation Pipeline...


--- Question 1/6 ---
Q: Who is the wife of the actor who played 'Neo' in The Matrix, and what is her birth year?
🛠️  Action: wikipedia -> {'query': 'Keanu Reeves wife birth year'}
👁️  Observation: Page: Keanu Reeves
Summary: Keanu Charles Reeves ( kee-AH-noo; born September 2, 1964) is a Canadian actor and musician. The recipient of numerous accolades in a career on screen spanning four decades...
🛠️  Action: wikipedia -> {'query': 'Keanu Reeves partner'}
👁️  Observation: Page: Good Fortune (film)
Summary: Good Fortune is a 2025 American comedy film written and directed by Aziz Ansari in his feature directorial debut. Seth Rogen, Ansari, Keke Palmer, Sandra Oh and Kean...
💭 Thought/Answer: The information about Keanu Reeves' wife or partner is not available in the provided Wikipedia pages.

Ground Truth: alexandra grant
EM Score: 0

--- Question 2/6 ---
Q: Who was the childhood best friend of the oldest person alive in 1850?
🛠️  Act